# Train ToM Steering Vector using CAA (Contrastive Activation Addition)

This notebook trains a steering vector using the CAA approach from the BigToM paper:

**Key Differences from Previous Approach:**
- Uses (prompt, cp, cn) triplets where completions are ONLY perceptual statements
- NO persona wrapping
- NO question/answer pairs in training
- Captures the representational difference between "witnessing" vs "not witnessing" events

**Dataset:**
- 1000 training triplets from simple witnessing scenarios (0_ prefix only)
- Positive completion: "Protagonist sees/witnesses the event"
- Negative completion: "Protagonist does not see/witness the event"
- Context is identical between positive and negative

**Model:** google/gemma-3-4b-it

## 1. Setup & Installation

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Clone the Cogni_map repository (if running in Colab)
import os
if not os.path.exists('Cogni_map'):
    !git clone https://github.com/ChuloIva/Cogni_map.git
    %cd Cogni_map

In [ ]:
# Install dependencies
!pip install -q transformers torch accelerate sentencepiece
!pip install -q hatchling
!pip install -q ToM/repeng/

# Verify installation
import sys
sys.path.insert(0, 'ToM/repeng')

try:
    from repeng import ControlVector, ControlModel, DatasetEntry
    print("✓ repeng successfully imported!")
except ImportError as e:
    print(f"✗ Import failed: {e}")
    print("\nTrying alternative installation...")
    !pip install -q numpy>=1.26.4 scikit-learn>=1.4.0 tqdm>=4.66.1 gguf>=0.13.0
    from repeng import ControlVector, ControlModel, DatasetEntry
    print("✓ repeng imported via sys.path!")

In [ ]:
# Optional: Mount Google Drive to save vectors permanently (Colab only)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    !mkdir -p /content/drive/MyDrive/tom_steering_vectors
    print("✓ Google Drive mounted")
except:
    print("Not running in Colab, skipping Drive mount")

## 2. Load Model (Gemma-3-4B)

In [ ]:
import json
import torch
import sys
from transformers import AutoModelForCausalLM, AutoConfig, Gemma3ForCausalLM, AutoTokenizer

# Ensure repeng is in path
if 'ToM/repeng' not in sys.path:
    sys.path.insert(0, 'ToM/repeng')

from repeng import ControlVector, ControlModel, DatasetEntry

print("✓ All imports successful!")

In [ ]:
# Model configuration
model_name = "google/gemma-3-4b-it"

print(f"Loading {model_name}...")

# Load config
config = AutoConfig.from_pretrained(model_name)

# Use bfloat16 for better numerical stability
print("Using bfloat16 for better numerical stability...")

# Load model
if hasattr(config, 'vision_config'):
    print("Detected vision-language model. Loading text-only version...")
    base_model = Gemma3ForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )
else:
    print("Loading standard causal LM...")
    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token_id = 0

print("Model loaded successfully!")
print(f"Device: {base_model.device}")
print(f"Dtype: {base_model.dtype}")

In [ ]:
# Wrap with ControlModel for steering
print("Setting up ControlModel...")

# For multimodal Gemma3, layers are at model.language_model.layers
if hasattr(base_model, 'language_model') and hasattr(base_model.language_model, 'layers'):
    print(f"Detected multimodal Gemma3 architecture")
    base_model.repeng_layers = base_model.language_model.layers
    num_layers = len(base_model.language_model.layers)
    base_model.config.num_hidden_layers = num_layers
elif hasattr(base_model, 'model') and hasattr(base_model.model, 'layers'):
    print(f"Detected standard architecture")
    base_model.repeng_layers = base_model.model.layers
    num_layers = len(base_model.model.layers)
else:
    raise ValueError("Could not find model layers!")

print(f"Total layers: {num_layers}")

# Use layers -4 to -20 (last 17 layers)
layer_ids = list(range(-4, -21, -1))
print(f"Wrapping layers: {layer_ids}")

model = ControlModel(base_model, layer_ids)

actual_layer_ids = [i if i >= 0 else num_layers + i for i in layer_ids]
print(f"Actual layer indices: {actual_layer_ids}")
print(f"✓ ControlModel initialized successfully!")

## 3. Load CAA Training Data

In [ ]:
# Load the CAA training triplets
with open("ToM/data/datagen/caa_training_data.json") as f:
    training_data = json.load(f)

# Load metadata
with open("ToM/data/datagen/caa_training_metadata.json") as f:
    metadata = json.load(f)

print(f"Loaded {len(training_data)} CAA training triplets")
print(f"\nMetadata:")
print(f"  Random seed: {metadata['random_seed']}")
print(f"  Total training examples: {metadata['num_examples_total']}")
print(f"  Remaining for evaluation: {metadata['remaining_for_eval']}")

print(f"\nExample CAA triplet:")
example = training_data[0]
print(f"Prompt: {example['prompt'][:150]}...")
print(f"Positive (sees): {example['positive_completion']}")
print(f"Negative (doesn't see): {example['negative_completion']}")

In [ ]:
# Convert to DatasetEntry format for repeng
# For CAA: positive = prompt + "sees", negative = prompt + "doesn't see"
dataset = [
    DatasetEntry(
        positive=f"{triplet['prompt']} {triplet['positive_completion']}",
        negative=f"{triplet['prompt']} {triplet['negative_completion']}"
    )
    for triplet in training_data
]

print(f"Created {len(dataset)} DatasetEntry objects for training")
print(f"\nExample DatasetEntry:")
print(f"Positive: {dataset[0].positive[:200]}...")
print(f"Negative: {dataset[0].negative[:200]}...")

## 4. Train the CAA ToM Vector

In [ ]:
# Train the steering vector using CAA approach
print("Training CAA ToM steering vector...")
print("This may take several minutes...\n")

model.reset()  # Always reset before training
caa_tom_vector = ControlVector.train(
    model, 
    tokenizer, 
    dataset, 
    method='pca_center'
)

print("\n✓ Training complete!")
print(f"Vector contains directions for {len(caa_tom_vector.directions)} layers")

## 5. Export the Vector

In [ ]:
# Export the vector
vector_path = "tom_caa_forward_belief.gguf"
caa_tom_vector.export_gguf(vector_path)
print(f"✓ Exported to: {vector_path}")

# Also save to Google Drive (if mounted)
try:
    import shutil
    drive_path = f"/content/drive/MyDrive/tom_steering_vectors/{vector_path}"
    shutil.copy(vector_path, drive_path)
    print(f"✓ Also saved to Google Drive: {drive_path}")
except Exception as e:
    print(f"Could not save to Google Drive: {e}")

## 6. Test the Vector

In [ ]:
# Helper function for generation
def generate_text(prompt, model, tokenizer, max_new_tokens=128):
    """
    Generate text from the model.
    """
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    
    output = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        repetition_penalty=1.1
    )
    
    return tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
# Test prompt: Classic false belief scenario
test_prompt = """Sarah puts her toy in the red box and leaves the room.
While she's gone, John moves the toy to the blue box.
When Sarah returns, where will she look for her toy?

Answer:"""

print("Testing CAA ToM steering vector...\n")
print("="*80)

# Baseline (no steering)
print("\n[BASELINE - No Steering]")
model.reset()
baseline_output = generate_text(test_prompt, model, tokenizer, max_new_tokens=50)
print(baseline_output)

# With positive ToM steering
print("\n" + "="*80)
print("\n[WITH CAA ToM STEERING - Strength: 1.5]")
print("Expected: Should recognize Sarah will look in red box (her false belief)")
print("-"*80)
model.set_control(caa_tom_vector, coeff=1.5)
steered_output = generate_text(test_prompt, model, tokenizer, max_new_tokens=50)
print(steered_output)

# With negative ToM steering (anti-ToM)
print("\n" + "="*80)
print("\n[ANTI-ToM STEERING - Strength: -2.0]")
print("Expected: May incorrectly say blue box (confusing reality with belief)")
print("-"*80)
model.set_control(caa_tom_vector, coeff=-2.0)
anti_steered_output = generate_text(test_prompt, model, tokenizer, max_new_tokens=50)
print(anti_steered_output)

# Reset model
model.reset()
print("\n" + "="*80)

## 7. Test with Your Own Scenarios

In [ ]:
# Try your own ToM scenario
custom_prompt = """Emma believes the meeting is at 3pm, but it was changed to 2pm.
She wasn't informed about the change. What time will Emma show up?

Answer:"""

print("Custom test:")
print("="*80)
print("\n[WITH CAA ToM STEERING]")
model.set_control(caa_tom_vector, coeff=1.5)
result = generate_text(custom_prompt, model, tokenizer, max_new_tokens=100)
print(result)

print("\n" + "="*80)
print("\n[WITHOUT CAA ToM STEERING]")
model.reset()
result = generate_text(custom_prompt, model, tokenizer, max_new_tokens=100)
print(result)

model.reset()

## 8. Download Vectors (Colab Only)

In [ ]:
# Download the vector file (Colab only)
try:
    from google.colab import files
    import os
    
    gguf_files = [f for f in os.listdir('.') if f.endswith('.gguf')]
    
    print(f"Found {len(gguf_files)} vector file(s):")
    for f in gguf_files:
        print(f"  - {f}")
    
    print("\nDownloading...")
    for f in gguf_files:
        files.download(f)
        print(f"Downloaded: {f}")
    
    print("\nAll vectors downloaded!")
except:
    print("Not running in Colab, skipping download")

## Notes

**CAA Approach vs. Traditional Approach:**
- Traditional: Uses persona wrapping + full Q&A pairs
- CAA: Uses ONLY perceptual statements as contrastive completions
- Advantage: Captures internal representation of perspective-taking without training on Q&A
- More generalizable and elegant than ITI probes

**Vector Strength (coeff parameter):**
- Start with values between -2.5 and 2.5
- Positive values: Enhance ToM capabilities (better perspective-taking)
- Negative values: Reduce ToM capabilities (confuse beliefs with reality)
- Typical good range: 1.0 to 2.0

**Training Data:**
- 1000 CAA triplets used for training (from 0_ prefix - simple witnessing scenarios)
- Examples remaining for evaluation in 0_ prefix conditions
- Training examples are excluded from downstream evaluation

**How to Use This Vector:**
```python
from repeng import ControlVector, ControlModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load model
model = AutoModelForCausalLM.from_pretrained("google/gemma-3-4b-it")
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-4b-it")
model = ControlModel(model, list(range(-4, -21, -1)))

# Load the vector
tom_vector = ControlVector.import_gguf("tom_caa_forward_belief.gguf")

# Apply the vector
model.set_control(tom_vector, coeff=1.5)

# Generate with ToM enhancement
# ... your generation code ...

# Reset when done
model.reset()
```